# Chapter 10 live coding — Fashion MNIST MLP in PyTorch

**Reference:** Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*, Ch. 10, “Building an Image Classifier with PyTorch”.

> Classroom adaptation: the architecture follows the chapter's Fashion-MNIST MLP (`784 → 300 → 100 → 10`). `FAST_MODE=True` uses a smaller training subset so the live demo fits in ~30 minutes; set it to `False` for the full 55k/5k split used in the book.

**This version adds an optional 15-minute Optuna extension after the core PyTorch live path.** 


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms.v2 as T
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", device)


## 0. Data — already prepared

Géron uses Fashion MNIST through TorchVision. Images arrive as `[C, H, W] = [1, 28, 28]`, while a mini-batch has shape `[B, 1, 28, 28]`.

The preprocessing below converts images to `float32` tensors and scales pixels to `[0, 1]`.


In [ ]:
to_tensor = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=to_tensor
)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=to_tensor
)

generator = torch.Generator().manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000], generator=generator
)

# Classroom shortcut: keep the same task/model, reduce only the amount of data.
FAST_MODE = True
if FAST_MODE:
    train_data = Subset(train_data, range(12_000))
    valid_data = Subset(valid_data, range(2_000))

BATCH_SIZE = 128 if FAST_MODE else 32
pin = device.type == "cuda"

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin
)
valid_loader = DataLoader(
    valid_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin
)
test_loader = DataLoader(
    test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin
)

class_names = train_and_valid_data.classes
print(f"train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}")
print(class_names)


## 1. Inspect one mini-batch

Before defining the network, inspect the tensor shapes that the model will receive.


In [ ]:
X_batch, y_batch = next(iter(train_loader))
print("X_batch:", X_batch.shape, X_batch.dtype)
print("y_batch:", y_batch.shape, y_batch.dtype)
print("first labels:", y_batch[:8].tolist())

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, image, label in zip(axes, X_batch[:6], y_batch[:6]):
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


## 2. Build the classifier — **LIVE**

Géron's classifier is a custom `nn.Module` containing an `nn.Sequential` stack:

$[1,28,28] \rightarrow 784 \rightarrow 300 \rightarrow 100 \rightarrow 10$.

Questions to ask while filling it in:

- Why is `Flatten` needed?
- Where do ReLUs go?
- Why are there **10** outputs?
- Why is there **no Softmax** in the final layer?


In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifier(
    n_inputs=28 * 28,
    n_hidden1=300,
    n_hidden2=100,
    n_classes=10,
).to(device)

model


## 3. Loss + one forward pass — **LIVE**

For exclusive multiclass classification, use:

```python
nn.CrossEntropyLoss()
```

Important: it expects **logits**, so we do **not** add a Softmax layer to the model.

Predict first:

- shape of the logits for a batch of size `B`;
- what `argmax(dim=1)` returns;
- why the targets can remain integer class indices.


In [ ]:
criterion = nn.CrossEntropyLoss()

X_batch, y_batch = next(iter(train_loader))
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

logits = model(X_batch)
loss = criterion(logits, y_batch)

print("logits shape:", logits.shape)
print("loss:", loss.item())


## 4. Optimizer — **LIVE**

The optimizer owns the parameter-update rule. Create it **after** moving the model to the accelerator.

For a short live demo we use SGD, matching the Chapter 10 discussion.

> The learning rate below is chosen for a short classroom run, not as a claim of an optimal hyperparameter.


In [ ]:
for p in model.parameters():
    print(p.shape)


In [ ]:
for name, p in model.named_parameters():
    print(name, p.shape)


In [ ]:
model.mlp[1].bias.data


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)


## 5. One training step — **LIVE, line by line**

This is the most important cell of the notebook.

Put these operations in the correct order:

1. forward pass;
2. compute loss;
3. backpropagation;
4. update parameters;
5. clear accumulated gradients.

Then inspect whether the loss is a scalar and whether gradients appeared.


In [ ]:
X_batch, y_batch = next(iter(train_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

logits = model(X_batch)                 # 1. forward
loss = criterion(logits, y_batch)       # 2. scalar objective
loss.backward()                         # 3. compute gradients
optimizer.step()                        # 4. update parameters
optimizer.zero_grad()                   # 5. clear accumulated gradients

print("loss:", loss.item())
first_weight = next(model.parameters())
print("gradient after zero_grad:", first_weight.grad)


## 6. Mini-batch training loop — **LIVE**

Now generalize the single step to all batches and epochs.

The essential PyTorch pattern is:

```text
model.train()
for batch:
    forward → loss → backward → step → zero_grad
```

The helper for validation is intentionally already provided below.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()

    return running_loss / len(loader)


## 7. Validation — helper already prepared

Evaluation is secondary to today's learning objective, so this function is given.

Notice the two important differences from training:

- `model.eval()`;
- no gradient tracking.

Accuracy is used here as an **evaluation metric**, not as the differentiable training loss.


In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        predictions = logits.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.numel()

    return correct / total


## 8. Train briefly — **LIVE**

For the live session, 2–3 epochs are enough to observe learning.

With `FAST_MODE=False`, you can reproduce the full dataset split and train longer after class.


In [ ]:
N_EPOCHS = 3 if FAST_MODE else 5

for epoch in range(N_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_acc = evaluate_accuracy(model, valid_loader)
    print(
        f"epoch {epoch + 1:02d}/{N_EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val acc {val_acc:.3%}"
    )


## 9. From logits to predictions and probabilities — **LIVE if time**

During training we feed **logits** directly to `CrossEntropyLoss`.

At inference:

- class prediction: `argmax(logits)`;
- probabilities: `softmax(logits)`.

This is exactly why Softmax does not belong inside this classifier's output layer during training.


In [ ]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:6].to(device)
y_new = y_new[:6]

with torch.no_grad():
    logits = model(X_new)
    y_pred = logits.argmax(dim=1)
    y_proba = torch.softmax(logits, dim=1)

print("true:", [class_names[i] for i in y_new.tolist()])
print("pred:", [class_names[i] for i in y_pred.cpu().tolist()])
print("confidence:", y_proba.max(dim=1).values.cpu())


## 10. The mental model to leave on screen

A PyTorch classifier is not “magic”. The core is:

$X \xrightarrow{\text{model}} \text{logits} \xrightarrow{\text{loss}} L \xrightarrow{\text{backward}} \nabla_\theta L \xrightarrow{\text{optimizer.step}} \theta'$.

**Five lines to remember:**

```python
logits = model(X)
loss = criterion(logits, y)
loss.backward()
optimizer.step()
optimizer.zero_grad()
```

### Optional 60-second questions

1. Why does the final layer have 10 units?
2. Why is there no Softmax inside the model?
3. What would happen if we forgot `optimizer.zero_grad()`?
4. Why do we call `model.eval()` for validation?
5. Why can accuracy be used for validation but not as our gradient-descent objective?

**Source:** Géron, Ch. 10, especially “Building an Image Classifier with PyTorch” and the chapter's mini-batch training loop.


# 11.  OPTUNA: 15-minute hyperparameter search

This section is a **separate 15-minute extension**. The core PyTorch lesson above is unchanged.

### Learning objective

Replace two manual choices with a small validation-driven search:

- learning rate `lr` — optimization hyperparameter;
- first hidden-layer width `n_hidden` — architecture hyperparameter.

The loop is:

$\text{sample hyperparameters} \rightarrow \text{build fresh model} \rightarrow \text{train briefly} \rightarrow \text{validation accuracy} \rightarrow \text{next trial}$.

### Live timing

| Time | Action |
|---:|---|
| 0–2 min | Why `lr=0.1` and `300` were manual choices |
| 2–5 min | Define the search space |
| 5–10 min | Build `objective(trial)` |
| 10–12 min | Run 5 trials |
| 12–14 min | Inspect `best_params` and trial table |
| 14–15 min | Validation vs test; mention pruning as after-class |


In [ ]:
!pip -q install optuna


In [ ]:
import optuna

print("Optuna version:", optuna.__version__)


### 11.1 Define the search space

Instead of choosing these values manually, each Optuna `trial` proposes them.

- `lr` is sampled on a **logarithmic scale**, because useful learning rates often span orders of magnitude.
- `n_hidden` is sampled as an integer width for the first hidden layer.

We deliberately tune only **two** hyperparameters during the live demo. The purpose is to understand the workflow, not to launch a large search.


In [ ]:
SEARCH_EPOCHS = 2
N_TRIALS = 5


def objective(trial):
    # 1. Hyperparameters proposed by Optuna
    lr = trial.suggest_float("lr", 1e-3, 3e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 64, 300)

    # 2. Fresh experiment for every trial
    torch.manual_seed(42)
    trial_model = ImageClassifier(
        n_inputs=28 * 28,
        n_hidden1=n_hidden,
        n_hidden2=100,
        n_classes=10,
    ).to(device)

    trial_optimizer = torch.optim.SGD(
        trial_model.parameters(),
        lr=lr,
    )
    trial_criterion = nn.CrossEntropyLoss()

    # Use the same batch order across trials to make the classroom
    # comparison less noisy.
    trial_train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=pin,
        generator=torch.Generator().manual_seed(42),
    )

    # 3. Very short training: enough to demonstrate HPO, not to claim an optimum
    best_val_acc = 0.0
    for epoch in range(SEARCH_EPOCHS):
        train_one_epoch(
            trial_model,
            trial_train_loader,
            trial_criterion,
            trial_optimizer,
        )
        val_acc = evaluate_accuracy(trial_model, valid_loader)
        best_val_acc = max(best_val_acc, val_acc)

    # 4. Optuna will maximize this validation score
    return best_val_acc


### 11.2  Run the study

`direction="maximize"` is used because the objective returns **validation accuracy**.

A seeded TPE sampler makes the classroom demo easier to reproduce. Five trials are intentionally few: this is a live demonstration, not a production search.


In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
)

study.optimize(objective, n_trials=N_TRIALS)


### 11.3  Inspect what Optuna found

Important wording for the class:

> This is the **best configuration among the trials we ran**, not proof of a globally optimal model.


In [ ]:
print(f"best validation accuracy: {study.best_value:.3%}")
print("best hyperparameters:", study.best_params)

trials = study.trials_dataframe(
    attrs=("number", "value", "params", "state")
)
trials


### 11.4  The experimental protocol matters

Hyperparameters are selected using the **validation set**, not the test set:

$\text{training} \rightarrow \text{fit model parameters}$  
$\text{validation} \rightarrow \text{choose hyperparameters}$  
$\text{test} \rightarrow \text{final evaluation only}$

Do **not** call `evaluate_accuracy(..., test_loader)` inside `objective()`. Doing so would leak test information into model selection.

For a real experiment, after selecting the hyperparameters we would typically rebuild the model, train it properly (often using train + validation data according to a pre-defined protocol), and evaluate the test set once.


# 12. Chapter 11 live extension: robust training

This section starts from the Fashion-MNIST classifier built above and keeps the **same data and task**. The goal is to modify the training recipe step by step, so that each change has a clear role.

### Live integration map

| Activity | Where it belongs in the original complete notebook | What to remove / replace during live coding | Time |
|---|---|---|---:|
| He/Kaiming initialization | immediately after **Section 2 — Build the classifier** | replace the default `nn.Linear` initialization by `model.apply(use_he_init)` | 5 min |
| BatchNorm | replace the model definition from **Section 2** | replace `Linear → ReLU` blocks by `Linear(bias=False) → BatchNorm1d → ReLU` | 8 min |
| AdamW | **Section 4 — Optimizer** | replace `torch.optim.SGD(...)` | 5 min |
| OneCycleLR | **Section 6 — Mini-batch training loop** | replace the old training helper with a scheduler-aware version; call `scheduler.step()` after every `optimizer.step()` | 9 min |
| Dropout | back in the model definition from **Section 2** | insert `Dropout` after hidden activations, then recreate model, optimizer and scheduler | 4 min |
| Integrated run | replaces **Section 8 — Train briefly** for this Chapter 11 session | train the final recipe and compare validation accuracy with the baseline | 9 min |

**Teaching strategy:** do not retrain fully after every modification. Inspect the local effect, then perform one integrated training run at the end.


## 12.0 Keep the original trained model as the baseline

**Where:** just before starting the Chapter 11 modifications.

**Replace/remove:** nothing. We only save its validation score before redefining `model`.

This gives us a reference point without touching the test set.


In [ ]:
baseline_val_acc = evaluate_accuracy(model, valid_loader)
print(f"baseline validation accuracy: {baseline_val_acc:.3%}")


## 12.1 Activity 1 — He/Kaiming initialization

**Where in the original notebook:** immediately after Section 2, once the model exists.

**What changes:** nothing in the architecture. We explicitly initialize every `nn.Linear` layer instead of relying on its default initialization.

During the live session, add the helper and call `model.apply(use_he_init)` right after constructing the model.


In [ ]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight, nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)

# Fresh model only for inspecting initialization.
torch.manual_seed(42)
model_he = ImageClassifier(
    n_inputs=28 * 28,
    n_hidden1=300,
    n_hidden2=100,
    n_classes=10,
).to(device)
model_he.apply(use_he_init)

print("first layer weight std:", model_he.mlp[1].weight.std().item())


## 12.2 Activity 2 — Replace the baseline architecture with BatchNorm

**Where in the original notebook:** replace the model definition in Section 2.

**Remove/replace:** replace `Linear → ReLU` with `Linear(bias=False) → BatchNorm1d → ReLU`.

The hidden-layer biases are disabled because BatchNorm already has learnable shift parameters. The final output layer remains unchanged.


In [ ]:
class ImageClassifierBN(nn.Module):
    def __init__(self, n_inputs=28 * 28, n_hidden1=300,
                 n_hidden2=100, n_classes=10):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1, bias=False),
            nn.BatchNorm1d(n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2, bias=False),
            nn.BatchNorm1d(n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifierBN().to(device)
model.apply(use_he_init)
model


In [ ]:
# Useful inspection point for the BatchNorm discussion.
bn = model.mlp[2]
print("BatchNorm momentum:", bn.momentum)
print("running mean shape:", tuple(bn.running_mean.shape))


## 12.3 Activity 3 — Replace SGD with AdamW

**Where in the original notebook:** Section 4 — Optimizer.

**Remove:** `optimizer = torch.optim.SGD(model.parameters(), lr=0.1)`.

**Replace with:** AdamW, introducing explicit weight decay as a regularization control.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print(optimizer)


## 12.4 Activity 4 — Replace the training helper to support OneCycleLR

**Where in the original notebook:** Section 6 — Mini-batch training loop.

**Replace:** the original `train_one_epoch(...)` for this session with the scheduler-aware version below.

The key change is the ordering inside each mini-batch: `backward → optimizer.step() → scheduler.step()`.

For OneCycleLR, one scheduler step corresponds to one optimizer update, not one epoch.


In [ ]:
CH11_EPOCHS = 5 if FAST_MODE else 8

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    epochs=CH11_EPOCHS,
    steps_per_epoch=len(train_loader),
)


def train_one_epoch_ch11(model, loader, criterion, optimizer,
                         scheduler=None, lr_history=None):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()

        optimizer.step()
        if scheduler is not None:
            scheduler.step()
            if lr_history is not None:
                lr_history.append(optimizer.param_groups[0]["lr"])

        running_loss += loss.item()

    return running_loss / len(loader)


## 12.5 Activity 5 — Add Dropout to the hidden representation

**Where in the original notebook:** return to the model definition in Section 2.

**Insert:** `nn.Dropout(0.2)` after each hidden activation.

**Important replacement rule:** once the architecture changes, create a **new model**, then recreate the optimizer and the scheduler. The old optimizer still points to the old model parameters.


In [ ]:
class ImageClassifierRegularized(nn.Module):
    def __init__(self, n_inputs=28 * 28, n_hidden1=300,
                 n_hidden2=100, n_classes=10, dropout=0.2):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1, bias=False),
            nn.BatchNorm1d(n_hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden1, n_hidden2, bias=False),
            nn.BatchNorm1d(n_hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifierRegularized(dropout=0.2).to(device)
model.apply(use_he_init)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    epochs=CH11_EPOCHS,
    steps_per_epoch=len(train_loader),
)


In [ ]:
# Small live check: stochastic in train mode, deterministic in eval mode.
X_check, _ = next(iter(valid_loader))
X_check = X_check[:8].to(device)

model.train()
train_equal = torch.allclose(model(X_check), model(X_check))

model.eval()
with torch.no_grad():
    eval_equal = torch.allclose(model(X_check), model(X_check))

print("same outputs in train mode?", train_equal)
print("same outputs in eval mode? ", eval_equal)


## 12.6 Activity 6 — Integrated training run

**Where in the original notebook:** replaces Section 8 — Train briefly for the Chapter 11 session.

At this point the complete recipe is He initialization + BatchNorm + AdamW/weight decay + OneCycleLR + Dropout.

Now train only once and compare the validation result with the original baseline.


In [ ]:
history_ch11 = {"train_loss": [], "val_acc": []}
lr_history = []

for epoch in range(CH11_EPOCHS):
    train_loss = train_one_epoch_ch11(
        model,
        train_loader,
        criterion,
        optimizer,
        scheduler=scheduler,
        lr_history=lr_history,
    )
    val_acc = evaluate_accuracy(model, valid_loader)

    history_ch11["train_loss"].append(train_loss)
    history_ch11["val_acc"].append(val_acc)

    print(
        f"epoch {epoch + 1:02d}/{CH11_EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val acc {val_acc:.3%}"
    )

print()
print(f"baseline validation accuracy: {baseline_val_acc:.3%}")
print(f"final validation accuracy:    {history_ch11['val_acc'][-1]:.3%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(history_ch11["train_loss"], marker="o")
axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch")
axes[0].grid(True)

axes[1].plot(lr_history)
axes[1].set_title("OneCycle learning rate")
axes[1].set_xlabel("optimizer step")
axes[1].set_ylabel("learning rate")
axes[1].grid(True)

plt.tight_layout()


## 12.7 End-of-session map: what did each intervention change?

| Mechanism | Changed component | Main question it addresses |
|---|---|---|
| He initialization | initial parameters | are signal/gradient scales reasonable at startup? |
| BatchNorm | hidden activations | can training remain numerically well behaved? |
| AdamW | optimizer/update rule | how are adaptive updates and weight decay applied? |
| OneCycleLR | learning-rate trajectory | how large should optimizer steps be over training? |
| Dropout | training-time representation | should the network rely less on individual hidden units? |

The important engineering habit is **controlled modification**: change one mechanism, know which part of the pipeline it affects, and keep train/validation/test roles separate.


# After class — proposed activities

These activities are deliberately outside the 40-minute live path. They are ordered from highest pedagogical value to more specialized extensions.

### A1 — Ablation study: which change actually helped?
Train the same model under a fixed seed and compare baseline SGD, + He initialization, + BatchNorm, + AdamW, + OneCycleLR, and + Dropout. Record final training loss, best validation accuracy, training time, and number of optimizer steps.

### A2 — Early stopping + best checkpoint
Extend the final training loop with `patience`, save the best `state_dict`, stop after repeated non-improvement, then reload the best checkpoint.

### A3 — Scheduler comparison
Keep model and optimizer fixed and compare constant LR, `ReduceLROnPlateau`, `CosineAnnealingLR`, and `OneCycleLR`. For each scheduler, identify exactly what one call to `scheduler.step()` represents.

### A4 — Gradient clipping
Insert `nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` after `loss.backward()` and before `optimizer.step()`. Log the gradient norm before and after clipping.

### A5 — MC Dropout
Start from `model.eval()`, reactivate only dropout modules, run multiple stochastic forward passes, and compare mean probability with prediction variability.

### A6 — Max-norm
After every `optimizer.step()`, project selected weight vectors back into an L2 ball. Contrast this with gradient clipping.

### A7 — AdamW parameter groups
Apply weight decay to weight matrices but not to bias or BatchNorm parameters. Compare with one global `weight_decay` value.

### A8 — Optuna with intermediate reporting and pruning
Use `trial.report()` at each validation epoch, then let a pruner stop weak configurations.

### A9 — Final test protocol
Only after all architecture, optimizer, scheduler and hyperparameter choices are fixed, rebuild/retrain the selected configuration and evaluate once on the test set.


## A2 starter — early stopping pseudocode

Add this logic immediately after computing the validation metric in the epoch loop. Save only the model with the best validation result.


In [ ]:
best_val_acc = -float("inf")
patience = 3
patience_counter = 0

# inside the epoch loop, after computing val_acc:
if val_acc > best_val_acc:
    best_val_acc = val_acc
    patience_counter = 0
    torch.save(model.state_dict(), "best_model.pt")
else:
    patience_counter += 1

if patience_counter >= patience:
    print("early stopping")
    # break

# after training:
# model.load_state_dict(torch.load("best_model.pt", weights_only=True))


## A4 starter — gradient clipping

Insert the clipping call between backpropagation and the optimizer update.


In [ ]:
loss.backward()
nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()


## A8 — Optuna: report intermediate results

`trial.report(value, step)` records an intermediate objective value. For neural-network training, a natural choice is one validation score per epoch. Reporting alone does not stop a trial; it exposes its progress to Optuna.


In [ ]:
def objective_with_report(trial):
    lr = trial.suggest_float("lr", 1e-3, 3e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 64, 300)

    model = ImageClassifier(
        n_inputs=28 * 28,
        n_hidden1=n_hidden,
        n_hidden2=100,
        n_classes=10,
    ).to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    best_val_acc = 0.0

    for epoch in range(5):
        train_one_epoch(model, train_loader, criterion, optimizer)
        val_acc = evaluate_accuracy(model, valid_loader)

        best_val_acc = max(best_val_acc, val_acc)

        # Tell Optuna how this trial is progressing at this epoch.
        trial.report(val_acc, step=epoch)

    return best_val_acc


### A8.1 — From reporting to pruning

Once intermediate validation values are available, ask the configured pruner whether the current trial should continue.


In [ ]:
trial.report(val_acc, step=epoch)

if trial.should_prune():
    raise optuna.TrialPruned()


**Key distinction:**

- `trial.report(...)` records an intermediate result;
- `trial.should_prune()` asks the configured pruner whether the trial should stop;
- `raise optuna.TrialPruned()` terminates that trial.


## A9 — Retrain the Optuna-selected configuration and evaluate the test set once

This closes the hyperparameter-selection loop. Keep this outside the live search: the test set is for final evaluation, not for selecting hyperparameters.


In [ ]:
best_lr = study.best_params["lr"]
best_hidden = study.best_params["n_hidden"]

# Rebuild a fresh model using the selected hyperparameters.
torch.manual_seed(42)
final_model = ImageClassifier(
    n_inputs=28 * 28,
    n_hidden1=best_hidden,
    n_hidden2=100,
    n_classes=10,
).to(device)

final_optimizer = torch.optim.SGD(final_model.parameters(), lr=best_lr)
final_criterion = nn.CrossEntropyLoss()

# Simple classroom protocol: retrain on the original training split.
# For a formal experiment, decide the train/validation recombination protocol
# before looking at the test result.
FINAL_EPOCHS = 5
for epoch in range(FINAL_EPOCHS):
    train_loss = train_one_epoch(
        final_model, train_loader, final_criterion, final_optimizer
    )
    print(f"final epoch {epoch + 1}/{FINAL_EPOCHS} | train loss {train_loss:.4f}")

final_test_acc = evaluate_accuracy(final_model, test_loader)
print(f"final test accuracy: {final_test_acc:.3%}")
